# 03 · Explainability: SHAP and LIME
* **SHAP** — Shapley values from cooperative game theory: each feature's fair share of the gap between this prediction and the average prediction. Additive (contributions + base value = prediction), consistent, gives both **global** (all seasons) and **local** (one season) views. `TreeExplainer` computes them exactly for tree ensembles.
* **LIME** — perturbs one input many times, fits a small weighted linear model around it, and reads its coefficients. Model-agnostic and intuitive, but **local only** and depends on random sampling and on how features are binned.

We explain the model chosen in notebook 02 (trained on ≤ 2015) on the **test** seasons, and compare SHAP vs LIME on the same cases.

In [ ]:
import sys, warnings, json
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib, shap
from src import config as C, explain as E
from src.features import TARGET, feature_list
from src.modeling import temporal_split
from src import train as T

df = T.load_table(); features = feature_list(df); train, test = temporal_split(df)
meta = json.loads((C.ARTIFACTS / "app_metadata.json").read_text())
best_name = meta["best_model"]
model = joblib.load(C.MODELS / "best_model_train_only.joblib")
print("Explaining:", best_name, "| tree model:", E.is_tree_pipeline(model))
# If the winner is not a tree model, also explain the best tree model so TreeExplainer is shown
cv = pd.read_csv(C.ARTIFACTS / "cv_results.csv")
tree_name = next(m for m in cv.model if m in ("XGBoost", "LightGBM", "Random Forest", "Gradient Boosting"))
tree_model = joblib.load(C.MODELS / f"{T.slug(tree_name)}.joblib")
print("Best tree model:", tree_name)
FIG = C.FIGURES

## 1. Global SHAP — which features drive yield forecasts?

In [ ]:
X_explain = test[features]
ex_tree = E.shap_explanation(tree_model, X_explain, features)
shap.plots.beeswarm(ex_tree, max_display=15, show=False)
plt.title(f"SHAP summary - {tree_name} (test seasons)"); plt.tight_layout()
plt.savefig(FIG / "shap_beeswarm.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
imp = E.shap_importance(ex_tree)
plt.figure(figsize=(8, 6)); imp.head(15)[::-1].plot.barh(color="#2a7ab9")
plt.xlabel("mean |SHAP| (kg/ha)"); plt.title("Global importance"); plt.tight_layout()
plt.savefig(FIG / "shap_bar.png", dpi=150); plt.show()
imp.round(1).to_frame("mean_abs_shap_kg_ha")

**Interpretation.**

- 

## 2. Dependence plots — shape of the effects and interactions

In [ ]:
for f, inter in [("rain_anom_pct", "season"), ("longest_dry_spell", "rain_anom_pct"), ("antecedent_rain_6m", "season")]:
    if f in features:
        j = features.index(f)
        vals = ex_tree[:, j]
        colour = test[inter] if pd.api.types.is_numeric_dtype(test[inter]) else (test[inter] == "Yala").astype(int)
        plt.figure(figsize=(6.5, 4))
        sc = plt.scatter(test[f], vals.values, c=colour, s=14, cmap="coolwarm", alpha=.8)
        plt.colorbar(sc, label=inter + (" (0=Maha,1=Yala)" if inter == "season" else ""))
        plt.axhline(0, c="k", lw=.7); plt.xlabel(f); plt.ylabel(f"SHAP value for {f}")
        plt.title(f"Dependence: {f}"); plt.tight_layout(); plt.savefig(FIG / f"shap_dep_{f}.png", dpi=150); plt.show()

**Interpretation.**

- 

## 3. SHAP by district group — is irrigation a buffer?

In [ ]:
j = features.index("rain_anom_pct")
shap_rain = pd.DataFrame({"district": test.district, "rain_anom": test.rain_anom_pct, "shap": ex_tree.values[:, j]})
slope = (shap_rain.groupby("district").apply(lambda g: np.polyfit(g.rain_anom, g.shap, 1)[0] * 10 if len(g) > 3 else np.nan)
         .sort_values())
plt.figure(figsize=(12, 4)); slope.plot.bar(color="#2a7ab9")
plt.ylabel("kg/ha per +10 % rain (SHAP slope)"); plt.title("Rain sensitivity learned by the model, by district")
plt.tight_layout(); plt.savefig(FIG / "shap_rain_sensitivity.png", dpi=150); plt.show()

## 4. Local explanations — three case studies

In [ ]:
p = tree_model.predict(test[features]); err = p - test[TARGET]
cases = {
    "Largest over-prediction": int(np.argmax(err)),
    "Driest season":           int(test.rain_anom_pct.idxmin()),
    "Typical season":          int((err.abs()).sub(err.abs().median()).abs().idxmin()),
}
case_df = test.loc[list(cases.values()), ["district", "season", "season_year", TARGET, "rain_anom_pct", "fertilizer_crisis"]]
case_df.insert(0, "case", list(cases.keys())); case_df["predicted"] = p[list(cases.values())].round(0)
case_df

In [ ]:
for label, i in cases.items():
    r = test.iloc[i]
    shap.plots.waterfall(ex_tree[i], max_display=10, show=False)
    plt.title(f"SHAP - {label}: {r.district} {r.season} {r.season_year}  (actual {r[TARGET]:.0f})")
    plt.tight_layout(); plt.savefig(FIG / f"shap_waterfall_{T.slug(label)}.png", dpi=150, bbox_inches="tight"); plt.show()

## 5. LIME on the same cases

In [ ]:
bundle = E.lime_explainer(tree_model, train[features], features)
lime_exps = {}
for label, i in cases.items():
    exp = E.lime_explain_row(tree_model, train[features], test.iloc[[i]], features, explainer_bundle=bundle)
    lime_exps[label] = exp
    fig = exp.as_pyplot_figure(); fig.set_size_inches(8, 4.5)
    r = test.iloc[i]
    plt.title(f"LIME - {label}: {r.district} {r.season} {r.season_year}")
    plt.tight_layout(); plt.savefig(FIG / f"lime_{T.slug(label)}.png", dpi=150); plt.show()
    print(f"model prediction {tree_model.predict(test.iloc[[i]][features])[0]:.0f}  |  LIME local surrogate {exp.local_pred[0]:.0f}  |  surrogate R2 {exp.score:.2f}")

## 6. SHAP vs LIME — do they agree?

In [ ]:
for label, i in cases.items():
    print(label)
    display(E.compare_shap_lime(ex_tree[i], lime_exps[label], top=6))

In [ ]:
# LIME stability: rerun with different random seeds - SHAP (TreeExplainer) is deterministic
i = cases["Typical season"]
tops = []
for seed in range(5):
    e = E.lime_explain_row(tree_model, train[features], test.iloc[[i]], features, seed=seed)
    tops.append([c for c, _ in e.as_list()[:3]])
pd.DataFrame(tops, columns=["1st", "2nd", "3rd"], index=[f"seed {s}" for s in range(5)])

| | SHAP (TreeExplainer) | LIME |
|---|---|---|
| Scope | Global + local | Local only |
| Theory | Shapley values; additive, consistent | Local linear surrogate on perturbed samples |
| Determinism | Exact and repeatable for trees | Varies with random seed and binning |
| Output form | kg/ha contribution per feature | Weight per *condition* (e.g. `rain_anom_pct <= -20`) |
| Cost | Fast for trees; slow (sampling) for other models | Fast, any model |
| Weakness | Splits credit between correlated features | Neighbourhood definition is arbitrary; can be unstable |

## 7. Model-agnostic SHAP for the selected model (if it is not a tree model)

In [ ]:
if not E.is_tree_pipeline(model):
    ex_best = E.shap_explanation(model, test[features].head(40), features, background=train)
    shap.plots.bar(ex_best, max_display=12, show=False); plt.title(f"Permutation SHAP - {best_name} (40 seasons)")
    plt.tight_layout(); plt.savefig(FIG / "shap_bar_selected_model.png", dpi=150); plt.show()
else:
    print("Selected model is a tree model - sections 1-6 already explain it.")

## 8. Interpretation summary
1. **What drives the forecast overall:** …
2. **Weather effects and their shape:** …
3. **Irrigation as a buffer:** …
4. **Why the model failed in 2021/22:** the fertilizer shock is outside the training distribution; SHAP shows the forecast was built from history + normal weather, so no feature 'explained' the drop.
5. **SHAP vs LIME:** where they agree / disagree and why.